In [ ]:
import sys
sys.path.append('../src')
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
from matplotlib.legend_handler import Line2D
import matplotlib as mpl
import matplotlib.colors as mcolors

from scipy.ndimage import uniform_filter1d, gaussian_filter1d, convolve1d

from tqdm import tqdm
import PI_CI_functions as pci

In [ ]:
def helmholtz_filter1d(length_scale, truncate=5.0):
    """
    Applies a 1D Helmholtz-like exponential filter to the input signal.

    Parameters:
    - length_scale: the exponential decay length (analogous to sigma in Gaussian)
    - truncate: kernel width as a multiple of length_scale
    """
    # Define kernel
    radius = max(1, int(truncate * length_scale))
    x = np.arange(-radius, radius + 1)
    kernel = (1 / (2 * length_scale)) * np.exp(-np.abs(x) / length_scale)
    kernel /= np.sum(kernel)  # Normalize

    # Apply convolution
    return kernel

---
## Algorithmic gains

In [ ]:
# Parameters
L = 1 # length of the embryo
n_cells = 100
n_embryos = 5000

win_sizes = np.linspace(1,n_cells,15, dtype=int)

lamb = L * 0.7 # length scale of exponentially decaying mean profile
mu_S = 5.7     # height of the mean profile at x=0
delta_x = L / n_cells # typical cell-cell spacing

sigma_ref = 0.31

# Intrinsic and extrinsic noise levels. Values chose such that the toal noise level is fixed and ratio is sigma_ratio
sigma_tot = sigma_ref; sigma_ratio = 15.0
sigma_int = sigma_tot / np.sqrt(1 + sigma_ratio**2)
sigma_ext = sigma_tot * sigma_ratio / np.sqrt(1 + sigma_ratio**2)

covariance_kernel = 'SimpleExponential'
x_positions = np.linspace(0, L-delta_x, n_cells)

corr_lengths = np.logspace(-0.5, 3.5, 20)


derivative_g = - mu_S/(lamb) * np.exp(-np.arange(n_cells)/(lamb*(n_cells/L)))

PIs = np.zeros(len(corr_lengths))
Ixxstar_theory = np.zeros(len(corr_lengths))
CIs = np.zeros(len(corr_lengths))
Delta_PIs_RLP_theory = np.zeros(len(corr_lengths))
Delta_PIs_ALP_theory = np.zeros(len(corr_lengths))

PIs_norm = np.zeros(len(corr_lengths))                       # PI of profiles after div. normalization
PIs_conv =  np.zeros((len(corr_lengths), len(win_sizes)))    # PI of profiles after convolution

PIs_norm_from_conv = np.zeros((len(corr_lengths), len(win_sizes)))   #  convlution -> normalization
PIs_conv_from_norm = np.zeros((len(corr_lengths), len(win_sizes)))   #  normalization -> convolution 
PIs_norm_local_conv = np.zeros((len(corr_lengths), len(win_sizes)))  #  normalization constant compute locally


In [ ]:
for i_cor, corr_length in tqdm(enumerate(corr_lengths), total=len(corr_lengths)):

    G = pci.samples_from_gp_exponential_profile(x_positions[:, np.newaxis], covariance_kernel, corr_length,
                                                delta_x, sigma_ext, sigma_int, mu_S, lamb, n_embryos)
    MG = mu_S * np.mean(np.exp(-x_positions / lamb))

    ## compute true theoretical covariance
    cov = np.zeros((n_cells, n_cells))
    for i in range(n_cells):
        for j in range(n_cells):
            if covariance_kernel == 'SimpleExponential':
                cov[i,j] = (sigma_int**2 * (i==j) + sigma_ext**2 * np.exp(-np.abs(i-j)/corr_length)) * np.exp(-(i+j)*delta_x/lamb)
            elif covariance_kernel == 'SquaredExponential':
                cov[i,j] = (sigma_int**2 * (i==j) + sigma_ext**2 * np.exp(-(i-j)**2/(2*corr_length**2))) * np.exp(-(i+j)*delta_x/lamb)
            else:
                raise ValueError('covariance kernel not recognized')
            
    inv_pos_errs_local = np.zeros(n_cells)
    for i_cell in range(n_cells):
        inv_pos_errs_local[i_cell] = derivative_g[i_cell]**2 / cov[i_cell, i_cell] 

    Ixxstar_theory[i_cor] = np.log2(L/(np.sqrt(2*np.pi*np.e))) + 0.5 * np.mean(np.log2(inv_pos_errs_local))
    
    inv_cov = np.linalg.inv(cov)
    _,_,PIs[i_cor] = pci.compute_PI(G, 500)
    _,CIs[i_cor] = pci.compute_CI(G, cov=cov)

    n_neigh = n_cells

    invpos_errs = np.zeros(n_cells)
    Delta_PIs_ALP_theory[i_cor] = 0
    for i_cell in range(n_cells):
        start = max(0, i_cell - n_neigh)
        end = min(n_cells, i_cell + n_neigh + 1)  # note: end is non-inclusive
        S_indices = np.arange(start, end)
        i_in_S = np.where(S_indices == i_cell)[0][0]
        invpos_errs[i_cell] = np.dot(derivative_g[S_indices], np.dot(np.linalg.inv(cov[S_indices][:, S_indices]), derivative_g[S_indices]))

        ## ---
        Delta_PIs_ALP_theory[i_cor] += 0.5/n_cells * ( np.log2(cov[i_cell, i_cell]) + np.log2(np.linalg.inv(cov[S_indices][:, S_indices])[i_in_S, i_in_S])) 

    Delta_PIs_RLP_theory[i_cor] = np.log2(L/(np.sqrt(2*np.pi*np.e))) + 0.5* np.mean(np.log2(invpos_errs)) - Ixxstar_theory[i_cor]

    # -------------------- Algorithmic gains --------------- #

    G_norm = G * MG / np.mean(G, axis=1)[:,np.newaxis]
    _,_, PIs_norm[i_cor] = pci.compute_PI(G_norm, 500)

    for i_win, win_size in enumerate(win_sizes):
        # Convolution with exponential filter #######################
        H_filter = helmholtz_filter1d(win_size, truncate=20.0)
        G_conv = convolve1d(G, H_filter, axis=1)
       
        _,_, PIs_conv[i_cor, i_win] = pci.compute_PI(G_conv, 500)

        G_local_norm_conv = G / G_conv * MG
        _,_, PIs_norm_local_conv[i_cor, i_win] = pci.compute_PI(G_local_norm_conv, 500)


        G_helmholtz_from_norm = convolve1d(G_norm, H_filter, axis=1)
        _,_, PIs_conv_from_norm[i_cor, i_win] = pci.compute_PI(G_helmholtz_from_norm, 500)

        MG_helmholtz = np.mean(G_conv)
        G_norm_from_helmholtz = G_conv * MG_helmholtz / np.mean(G_conv, axis=1)[:,np.newaxis]

        _,_, PIs_norm_from_conv[i_cor, i_win] = pci.compute_PI(G_norm_from_helmholtz, 500)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(6., 2.), dpi = 250, sharey=True)
fig.subplots_adjust(wspace=0.05)

win_sizes_fractions = [win_sizes[i]/n_cells for i in range(len(win_sizes))]
# Normalize window sizes for color intensity
norm_win = plt.Normalize(min(win_sizes_fractions), max(win_sizes_fractions))

colors_win = cm.coolwarm(norm_win(win_sizes_fractions))  # Generate shades of red

optimals = [Delta_PIs_RLP_theory, Delta_PIs_ALP_theory, [Delta_PIs_RLP_theory, Delta_PIs_ALP_theory]]
algo_gains = [PIs_conv, PIs_norm_local_conv, PIs_norm_from_conv]

max_algos = [np.max(PIs_conv, axis=1), PIs_norm, np.max(PIs_norm_from_conv, axis=1)]

titles = ['Convolution', 'Divisive Norm.', 'Conv. + Norm.']

for ax, optimal, algo_gain, max_algo, title in zip(axs, optimals, algo_gains, max_algos, titles):
    if isinstance(optimal, list):
        for opt in optimal:
            ax.plot(CIs, opt, color='k', zorder=0, linestyle='-', lw=3, alpha=0.5)
    else:
        ax.plot(CIs, optimal, color='k', zorder=0, linestyle='-', lw=3, alpha=0.5)

    for i_win, win_size in enumerate(win_sizes):
        ax.plot(CIs, algo_gain[:, i_win] - PIs[0], color=colors_win[i_win], alpha=0.5, marker='o')

    ax.plot(CIs, max_algo - PIs[0], color='mediumseagreen', marker='o', alpha=.7, zorder=10)

    ax.set_title(title, fontsize=13)
    ax.axhline(y = 0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axhline(y = np.log2(n_cells) - PIs[0], linestyle='--', color='k', zorder=12, alpha=0.5)

    ax.set_xlabel('CI [bits]', fontsize=12)
    if ax == axs[0]:
        ax.set_ylabel(r'$\Delta PI$ [bits]', fontsize=12)
    ax.set_ylim(-.5,3.95)
    ax.set_xlim(np.min(CIs)-0.05, np.max(CIs)+0.05)

# add horizontal colorbar for window sizes
cbar_ax = fig.add_axes([0.38, -0.23, 0.5, 0.07])  # [left, bottom, width, height]
scalarmappaple_win = cm.ScalarMappable(norm=norm_win, cmap='coolwarm')
scalarmappaple_win.set_array([])
cbar = fig.colorbar(scalarmappaple_win, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Kernel radius / N', fontsize=13)
cbar.ax.tick_params(labelsize=13)

# make a legend with 1 line for optimal, and 1 for max_algo
legend_elements = [
    Line2D([0], [0], color='k', lw=3, label='RLP & ALP', alpha=0.5),
    Line2D([0], [0], marker='o', color='mediumseagreen', label='Max. Algo. Gain', markersize=6, linestyle='None')
]
plt.legend(handles=legend_elements, bbox_to_anchor=(-0.28, 1.2), loc='upper center', fontsize=8, framealpha=.85).set_zorder(15)

### Equivalent plots using heatmaps and contour plots

In [ ]:
vmin = min(np.min(PIs_conv - PIs[0]), np.min(PIs_norm_from_conv - PIs[0]), np.min(PIs_norm_local_conv - PIs[0]))
vmax = max(np.max(PIs_conv - PIs[0]), np.max(PIs_norm_from_conv - PIs[0]), np.max(PIs_norm_local_conv - PIs[0]))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(1, 3, figsize=(10, 3), dpi=250, sharex=True, sharey=True)

datasets = [PIs_conv, PIs_norm_local_conv, PIs_norm_from_conv]
titles = ['Convolution', 'Div. Norm.', 'Norm. after Conv.']

for ax, data, title in zip(axs, datasets, titles):
    pcm = ax.pcolormesh(win_sizes/n_cells, corr_lengths/n_cells, data - PIs[0], 
                        cmap='viridis', shading='auto', norm=norm)
    
    cs= ax.contour(win_sizes/n_cells, corr_lengths/n_cells, data - PIs[0], 
                   levels=7, colors='white', linewidths=1)
    
    ax.set_yscale('log')
    ax.set_xlabel('Kernel radius (fraction of N)')
    ax.set_title(title)
    if ax==axs[0]:
        ax.set_ylabel('Correlation length (fraction of N)')

    ### argmax
    argmax_PI = np.argmax(data, axis=1)
    max_pi_per_corr = data[np.arange(len(corr_lengths)), argmax_PI] - PIs[0]
    ax.scatter(win_sizes[argmax_PI]/n_cells, corr_lengths/n_cells, color = cm.viridis(norm(max_pi_per_corr + 0.2)), marker='*', edgecolors='k', s=60, zorder=10)
    

cbar = fig.colorbar(pcm, ax=axs, orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label(r'$\Delta PI$ [bits]', fontsize=10)